In [1]:
import pandas as pd

# Load the original weather dataset
df = pd.read_csv("GlobalWeatherRepository.csv")

print("Source dataset loaded successfully!")
print("Total source records:", len(df))

display(df.head())

Source dataset loaded successfully!
Total source records: 164499


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [2]:
# Select required columns for AtmoSync

required_columns = [
    "location_name",
    "latitude",
    "longitude",
    "temperature_celsius",
    "humidity"
]

atmosync_source = df[required_columns].copy()

print("Required columns selected successfully!")
print("Rows:", len(atmosync_source))
print("Columns:", len(atmosync_source.columns))

display(atmosync_source.head())

Required columns selected successfully!
Rows: 164499
Columns: 5


,location_name,latitude,longitude,temperature_celsius,humidity
0,Kabul,34.52,69.18,26.6,24
1,Tirana,41.33,19.82,19.0,94
2,Algiers,36.76,3.05,23.0,29
3,Andorra La Vella,42.50,1.52,6.3,61
4,Luanda,-8.84,13.23,26.0,89


In [3]:
# Create stable Container IDs

# Remove rows with missing required values
atmosync_source = atmosync_source.dropna().reset_index(drop=True)

# Select 100 different locations
container_source = atmosync_source.drop_duplicates(
    subset=["location_name"]
).head(100).copy()

# Create stable Container IDs
container_source["Container_ID"] = [
    "CONT_{:04d}".format(i)
    for i in range(1, len(container_source) + 1)
]

print("Containers created:", len(container_source))

display(container_source.head())

Containers created: 100


,location_name,latitude,longitude,temperature_celsius,humidity,Container_ID
0,Kabul,34.52,69.18,26.6,24,CONT_0001
1,Tirana,41.33,19.82,19.0,94,CONT_0002
2,Algiers,36.76,3.05,23.0,29,CONT_0003
3,Andorra La Vella,42.50,1.52,6.3,61,CONT_0004
4,Luanda,-8.84,13.23,26.0,89,CONT_0005


In [4]:
# Check how many historical records are available for each location

location_counts = atmosync_source["location_name"].value_counts()

print("Location record count:")
display(location_counts.head(20))

print(
    "\nLocations with at least 50 records:",
    (location_counts >= 50).sum()
)

Location record count:


location_name
Sanaa           847
Kyiv            846
Malabo          846
Bern            846
Dakar           846
Bujumbura       846
Tokyo           846
Accra           846
Tashkent        846
N'djamena       846
Vatican City    846
Valletta        846
Baghdad         846
Warsaw          846
Asmara          846
Nicosia         845
Lilongwe        845
Singapore       845
Antananarivo    845
Bratislava      845
Name: count, dtype: int64


Locations with at least 50 records: 212


In [5]:
# Select 100 locations that have at least 50 historical records

eligible_locations = location_counts[location_counts >= 50].head(100).index

print("Eligible locations selected:", len(eligible_locations))

# Keep only the selected locations
selected_data = atmosync_source[
    atmosync_source["location_name"].isin(eligible_locations)
].copy()

print("Records available for selected locations:", len(selected_data))

display(selected_data.head())

Eligible locations selected: 100
Records available for selected locations: 84476


,location_name,latitude,longitude,temperature_celsius,humidity
0,Kabul,34.52,69.18,26.6,24
1,Tirana,41.33,19.82,19.0,94
3,Andorra La Vella,42.50,1.52,6.3,61
4,Luanda,-8.84,13.23,26.0,89
7,Yerevan,40.18,44.51,19.0,40


In [7]:
# Rebuild the final dataset with exactly 100 containers × 50 records

# Get 100 locations that each have at least 50 records
eligible_locations = (
    location_counts[location_counts >= 50]
    .head(100)
    .index
    .tolist()
)

# Create a clean Container ID mapping
container_mapping = {
    location: f"CONT_{i:04d}"
    for i, location in enumerate(eligible_locations, start=1)
}

# Select data for exactly these 100 locations
final_source = atmosync_source[
    atmosync_source["location_name"].isin(eligible_locations)
].copy()

# Take exactly 50 records from each location
final_source = (
    final_source
    .groupby("location_name", group_keys=False)
    .head(50)
    .copy()
)

# Assign Container IDs
final_source["Container_ID"] = final_source["location_name"].map(
    container_mapping
)

# Rename columns
atmosync_final = final_source.rename(columns={
    "temperature_celsius": "Temperature_C",
    "humidity": "Humidity_Percent",
    "latitude": "Latitude",
    "longitude": "Longitude"
})

# Create simulated observation timestamps
atmosync_final["Timestamp"] = pd.date_range(
    start="2026-09-01 00:00:00",
    periods=len(atmosync_final),
    freq="min"
)

# Keep required columns
atmosync_final = atmosync_final[
    [
        "Container_ID",
        "Timestamp",
        "Temperature_C",
        "Humidity_Percent",
        "Latitude",
        "Longitude"
    ]
].reset_index(drop=True)

print("Final AtmoSync dataset created!")
print("Total records:", len(atmosync_final))
print("Total containers:", atmosync_final["Container_ID"].nunique())

display(atmosync_final.head(10))

Final AtmoSync dataset created!
Total records: 5000
Total containers: 100


,Container_ID,Timestamp,Temperature_C,Humidity_Percent,Latitude,Longitude
0,CONT_0024,2026-09-01 00:00:00,26.6,24,34.52,69.18
1,CONT_0066,2026-09-01 00:01:00,19.0,94,41.33,19.82
2,CONT_0042,2026-09-01 00:02:00,6.3,61,42.50,1.52
3,CONT_0043,2026-09-01 00:03:00,26.0,89,-8.84,13.23
4,CONT_0046,2026-09-01 00:04:00,19.0,40,40.18,44.51
5,CONT_0082,2026-09-01 00:05:00,9.0,87,-35.28,149.22
6,CONT_0083,2026-09-01 00:06:00,16.0,63,48.20,16.37
7,CONT_0084,2026-09-01 00:07:00,17.0,68,40.40,49.88
8,CONT_0050,2026-09-01 00:08:00,36.0,33,26.24,50.58
9,CONT_0085,2026-09-01 00:09:00,38.4,31,23.72,90.41


In [8]:
# Verify records for each container

container_counts = (
    atmosync_final["Container_ID"]
    .value_counts()
    .sort_index()
)

print("Number of containers:", len(container_counts))
print("Minimum records per container:", container_counts.min())
print("Maximum records per container:", container_counts.max())

display(container_counts.head(10))

Number of containers: 100
Minimum records per container: 50
Maximum records per container: 50


Container_ID
CONT_0001    50
CONT_0002    50
CONT_0003    50
CONT_0004    50
CONT_0005    50
CONT_0006    50
CONT_0007    50
CONT_0008    50
CONT_0009    50
CONT_0010    50
Name: count, dtype: int64

In [9]:
# Final data quality checks

print("Missing values:")
display(atmosync_final.isnull().sum())

print("\nDuplicate rows:", atmosync_final.duplicated().sum())

print("\nDataset shape:", atmosync_final.shape)

print("\nData types:")
display(atmosync_final.dtypes)

Missing values:


Container_ID        0
Timestamp           0
Temperature_C       0
Humidity_Percent    0
Latitude            0
Longitude           0
dtype: int64


Duplicate rows: 0

Dataset shape: (5000, 6)

Data types:


Container_ID                object
Timestamp           datetime64[ns]
Temperature_C              float64
Humidity_Percent             int64
Latitude                   float64
Longitude                  float64
dtype: object

In [10]:
# Save the final AtmoSync dataset

output_file = "AtmoSync_Final_Data.csv"

atmosync_final.to_csv(output_file, index=False)

print("Final dataset saved successfully!")
print("File name:", output_file)
print("Total records:", len(atmosync_final))

Final dataset saved successfully!
File name: AtmoSync_Final_Data.csv
Total records: 5000
